In [2]:
# Import libraries
import pandas as pd
import numpy as np
import json
from openai import OpenAI
from dotenv import load_dotenv
import os

In [47]:
df = pd.read_csv("virma_routes_export.csv")

In [49]:
# Combine texts and check for pyhiinvaellus and toilet
def combine_texts(row, columns):
    # Combine text from specified columns
    text = ' '.join([str(row[col]) for col in columns if pd.notna(row[col])]).lower()

    # Pyhiinvaellus check
    key_word_pyh = 'pyhiinvaellus'
    pyhiinvaellus_flag = key_word_pyh in text

    # Toilet check
    keywords_toilet = ["wc", "kuivakäymälä", "käymälä", "huussi", "puucee", "vessa", "inva-wc"]
    has_toilet_flag = any(keyword in text for keyword in keywords_toilet)

    # Return results as dictionary
    return {'pyhiinvaellus': pyhiinvaellus_flag, 'has_toilet': has_toilet_flag}


In [50]:
text_columns = ['info_fi', "accessibil", "chall_clas", "shapeestim", "special"]

In [51]:
# Apply function to all rows to get toilet and pilgrimage columns

df_flags = df.apply(lambda row: pd.Series(combine_texts(row, text_columns)), axis=1)
df = pd.concat([df, df_flags], axis=1)

In [68]:
df[df["special"].notna()][["pyhiinvaellus", "special"]]

,pyhiinvaellus,special
12,True,Pyhiinvaellus
13,False,Velhoveden reitti
20,True,Pyhiinvaellus
21,True,Pyhiinvaellus
38,False,Teijo
39,False,Teijo
41,False,Teijo
44,False,Leader-kohde
46,False,Leader-kohde
48,False,Teijo


In [53]:
# Accessibility check via OpenAI API
load_dotenv(override=True)
api_key = os.getenv("GPT_KEY")

client = OpenAI(api_key=api_key)

def classify_accessibility(text: str) -> str:
    system_prompt = (
        "You are a classification assistant. "
        "Your job is to classify a description of a physical route or location "
        "into one of the following categories:\n\n"
        "1. esteellinen – not accessible for people with mobility impairments.\n"
        "2. esteetön – accessible (e.g., smooth paths, ramps, step-free access).\n"
        "3. vaativa esteetön – accessible but demanding; may require an assistant "
        "and/or special equipment such as an all-terrain wheelchair.\n\n"
        "If the text is unclear or no accessibility information is given, classify it as 'esteellinen'. "
        "Return ONLY one of these three labels exactly: "
        "esteellinen, esteetön, vaativa esteetön."
    )

    user_prompt = f'Text: "{text}"\n\nAccessibility class:'

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    answer = response.choices[0].message.content.strip().lower()

    # validation
    valid = ["esteellinen", "esteetön", "vaativa esteetön"]
    if answer not in valid:
        return "esteellinen"
    
    return answer

In [54]:
def combine_and_check(row):

    access = str(row['accessibil']) if pd.notna(row['accessibil']) else ""
    desc = str(row['info_fi']) if pd.notna(row['info_fi']) else ""
    chall_clas = str(row['chall_clas']) if pd.notna(row['chall_clas']) else ""

    text = f"{desc} {access} {chall_clas}".strip()
    
    return classify_accessibility(text)

In [55]:
df["accessibility_prediction"] = df.apply(combine_and_check, axis=1)

In [56]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [57]:
df[df["accessibility_prediction"]== "esteetön"][["gid", "name_fi", "lipas_type_name","info_fi", "chall_clas", "shapeestim", "accessibil", "accessibility_prediction"]]

,gid,name_fi,lipas_type_name,info_fi,chall_clas,shapeestim,accessibil,accessibility_prediction
45,283,Jeturkastin esteetön reitti 1,Retkeilyreitti,"Pyörätuolilla tai lastenrattailla kuljettavaksi suunniteltu sepelöity reitti Teijon kansallispuistossa. Pituus 0,6 km.",NaN,NaN,pyörätuolilla kulkeminen on mahdollista,esteetön
77,305460,Kirkkojärven reitti,Pyöräilyreitti,Reitti kulkee Someron keskustassa sijaitsevan Kirkkojärven yli. Järven pääsee kiertämään Härkälän kartanon lähellä olevan kävelysillan kautta. Reitin pituus 7 km.,Maastoltaan tasainen reitti. Ei juurikaan mäkiä. Osittain hiekkatietä osittain asfalttia.,Reitti on pääasiassa valtion tietä (sekä asfaltti että hiekka),NaN,esteetön
92,298,Jeturkastin esteetön reitti 2,Retkeilyreitti,"Pyörätuolilla tai lastenrattailla kuljettavaksi suunniteltu sepelöity reitti Teijon kansallispuistossa. Pituus 0,7 km.",NaN,NaN,pyörätuolilla kulkeminen on mahdollista,esteetön
111,140,Jeturkastin muinaispolku - esteetön osuus,Retkeilyreitti,Jeturkastin esteetön reittiosuus. Alue kuuluu Teijon kansallispuistoon.,NaN,NaN,NaN,esteetön
114,138,Kuhankuonon retkeilyreitistö - Kurjenpesän esteetön rantapolku,Retkeilyreitti,Esteetön polku Kurjenpesän luontotuvalta Savojärven rantaan ja laiturille. Pohjamaan on kansallispuistoa.,NaN,NaN,NaN,esteetön
132,425,Kumpulan jokipolku,Luontopolku,"Kumpulan jokipolku kulkee Paimionjoen upeissa maisemissa. Reitin varrella on myös tulipaikka, jonka läheisyydestä pääsee rantaan.",Helppokulkuinen,NaN,NaN,esteetön
140,312,Tuorlan luontopolku,Luontopolku,"Noin 1 km polku. Tuorlan luontopolkukokonaisuus, jossa mm. Helmeri Harakan luontopolku, joka on merkitty keltaisilla nuolilla ja harakkamerkeillä. Polun varrella on 10 lapsille suunnattua rastia. Myös Peikkopolku-nimellä tunnettu reitti.",helppo,NaN,Myös esteetön polku,esteetön
149,452,Kurjen pesän tulipaikan esteetön polku,Retkeilyreitti,Esteetön reitti Kurjenpesän tulipaikalle,NaN,NaN,NaN,esteetön
189,300,Kariholman esteetön reitti,Retkeilyreitti,"Pyörätuolilla tai lastenrattailla kuljettavaksi suunniteltu sepelöity reitti Teijon kansallispuistossa. Pituus 0,6 km.",NaN,NaN,pyörätuolilla kulkeminen on mahdollista,esteetön


In [58]:
df.loc[df["gid"] == 312, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 425, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 36, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 305460, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 50, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 239, "accessibility_prediction"] = "esteellinen"

In [59]:
df[df["accessibility_prediction"]== "vaativa esteetön"][["gid", "name_fi", "lipas_type_name","info_fi", "chall_clas", "shapeestim", "accessibil", "accessibility_prediction"]]

,gid,name_fi,lipas_type_name,info_fi,chall_clas,shapeestim,accessibil,accessibility_prediction
5,302307,Torpparin taival,Luontopolku,"Torpparin taival on Vuohensaaren luontopoluista helpoin. Luontopolku alkaa tanssilavan edessä olevalta opastustaululta ja johtaa näköalapaikalle, josta avautuu kaunis näköala Halikonlahdelle. Merkitty oranssein vinoneliöin.",helppo - lätt - easy,NaN,Soveltuu lastenrattaille ja liikuntarajoitteisille. Ei kuitenkaan täysin esteetön. Esteetön WC grillikatoksen lähellä.,vaativa esteetön
193,368,Vargbergetin polku,Luontopolku,"Reitin pituus 2,4 km. Reitti vie Vargbergetin näkötornille ja laavulle.Lähtöpiste Petsorintie 55, 21670 Parainen. Kalliolla kiertelevän reitin varrella luolia, siirtolohkareita ja näkötornista upea näkymä saaristoon.","korkeuseroja jonkin verran, muutamia haastavia kohtia kallion kiertolenkillä",Reitti on hyvässä kunnossa ja kallio-osuudella olevat juuri tehdyt oranssit opasteet ovat hyvät ja näkyvät.,Reitti kulkee alussa hakkuuaukean ja se osuus on helppokulkuinen. Näkötornille nousu on melko jyrkkä. Kalliota kiertävä reitti on paikoin hieman haastava.,vaativa esteetön
214,302322,Hirvensalon pyöräilyreitti,Pyöräilyreitti,Reitti kulkee Turun eteläpuolella sijaitsevan Hirvensalon saaren läpi. Hirvensalo on Turun ripeimmin kasvavia asuinalueita. Yhä kiihtyvästä rakennustahdista huolimatta saarelta löytyy merellistä luontoa ja idyllistä maaseutua.,"Reitti sisältää nousuja, eli jos mäkisyys arveluttaa, on sähköpyörä tälle reitille hyvä valinta.",NaN,NaN,vaativa esteetön
235,364,Mustikkapolku,Luontopolku,"2,8 km pitkä polku. Merkitty sinisillä merkinnöillä ja viitoilla. Maasto kalliota ja soistuvaa maata. Lähtöpisteessa (Brattnäsintie 73, 21630 Parainen) tilaa noin viidelle autolle.","keskitasoa - korkeuseroja on, mutta ei kovin jyrkkiä nousuja","Reitti on hyvin merkitty maastoon sinisillä viitoituksilla, joten eksymisen vaaraa ei ole. Parkkialueen vieressä on infotaulu, jossa koko reitti on kartalla.","Reitillä on jonkin verran korkeuseroa, joten hyvät jalkineet ovat tarpeen. Viitoitus on kuitenkin hyvässä kunnossa.",vaativa esteetön


In [77]:
df.loc[df["gid"] == 305462, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 164, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 368, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 364, "accessibility_prediction"] = "esteellinen"
df.loc[df["gid"] == 302322, "accessibility_prediction"] = "esteellinen"

In [61]:
df.shape

(237, 34)

In [79]:
import json

df['properties'] = df.apply(lambda row: json.dumps({
    'accessibile?': row['accessibility_prediction'],
    'toilet?': row['has_toilet'],
    'pyhiinvaellus?': row['pyhiinvaellus']
}), axis=1)

In [80]:
data_to_export = df.drop(columns=["accessibility_prediction", "has_toilet", "pyhiinvaellus"])
data_to_export.columns

Index(['gid', 'geom', 'name_fi', 'name_en', 'name_se', 'municipali',
       'subregion', 'region', 'info_fi', 'info_se', 'info_en', 'chall_clas',
       'accessibil', 'length_m', 'www_fi', 'www_se', 'www_en', 'telephone',
       'email', 'timestamp', 'upkeeper', 'upkeepclas', 'upkeepinfo',
       'shapeestim', 'munici_nro', 'subreg_nro', 'region_nro', 'special',
       'ski_route', 'lipas_type_name', 'lipas_type_code', 'properties'],
      dtype='object')

In [81]:
data_to_export.loc[
    data_to_export["properties"].apply(lambda x: json.loads(x)["accessibile?"] == "vaativa esteetön"),
    ["gid", "name_fi", "lipas_type_name", 'info_fi', "accessibil", "chall_clas", "shapeestim", "special", "properties"]
]

,gid,name_fi,lipas_type_name,info_fi,accessibil,chall_clas,shapeestim,special,properties
5,302307,Torpparin taival,Luontopolku,"Torpparin taival on Vuohensaaren luontopoluista helpoin. Luontopolku alkaa tanssilavan edessä olevalta opastustaululta ja johtaa näköalapaikalle, josta avautuu kaunis näköala Halikonlahdelle. Merkitty oranssein vinoneliöin.",Soveltuu lastenrattaille ja liikuntarajoitteisille. Ei kuitenkaan täysin esteetön. Esteetön WC grillikatoksen lähellä.,helppo - lätt - easy,NaN,NaN,"{""accessibile?"": ""vaativa esteet\u00f6n"", ""toilet?"": true, ""pyhiinvaellus?"": false}"


In [85]:
data_to_export.shape

(237, 32)

In [94]:
data_to_export[data_to_export["shapeestim"].notna()][["shapeestim"]].head(20)

,shapeestim
9,käyttökielto
40,Hyvässä kunnossa
54,Reitti on pääasiassa valtion tietä (sekä asfaltti että hiekka)
57,"Rakenteet kunnossa, polku myös"
66,Reitti on pääasiassa valtion tietä (sekä asfaltti että hiekka)
75,Reitti on suurimmaksi osaksi valtion tietä (sekä asfaltti että hiekka)
77,Reitti on pääasiassa valtion tietä (sekä asfaltti että hiekka)
118,erinomainen
125,"Sateen jälkeen kosteutta, keväällä sulamisvesiä, märällä kelillä liukkautta. Pitkospuita. Opastettu viitoituksin."
152,"Reitti on suhteellisen hyvin merkitty maastoon ja sitä on käytetty sen verran, että polku on muodostunut selväpiirteiseksi."


In [98]:
data_to_export = df.drop(columns=["special", "pyhiinvaellus", "has_toilet", "accessibility_prediction"])
data_to_export.columns

Index(['gid', 'geom', 'name_fi', 'name_en', 'name_se', 'municipali',
       'subregion', 'region', 'info_fi', 'info_se', 'info_en', 'chall_clas',
       'accessibil', 'length_m', 'www_fi', 'www_se', 'www_en', 'telephone',
       'email', 'timestamp', 'upkeeper', 'upkeepclas', 'upkeepinfo',
       'shapeestim', 'munici_nro', 'subreg_nro', 'region_nro', 'ski_route',
       'lipas_type_name', 'lipas_type_code', 'properties'],
      dtype='object')

In [100]:
data_to_export.to_csv("virma_lipas__routes_export.csv", index=False, na_rep="NULL")

In [102]:
data_to_export["ski_route"]

0      False
1      False
2      False
3      False
4      False
       ...  
232    False
233    False
234    False
235    False
236    False
Name: ski_route, Length: 237, dtype: bool

In [3]:
tarkistusdf = pd.read_csv("../../Data/virma_lipas__routes_export2.csv")

In [4]:
tarkistusdf

,gid,geom,name_fi,name_en,name_se,municipali,subregion,region,info_fi,info_se,...,upkeepclas,upkeepinfo,shapeestim,munici_nro,subreg_nro,region_nro,ski_route,lipas_type_name,lipas_type_code,properties
0,302297,"MULTILINESTRING ((222788 6734427, 222820 67344...",Tammimetsä,NaN,NaN,Mynämäki,Turku,Varsinais-Suomi,Saa poiketa merkityltä polulta,NaN,...,ajoittainen,NaN,NaN,503,023,02,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
1,302295,"MULTILINESTRING ((224084 6734475, 224049 67344...",Oikopolku,NaN,NaN,Mynämäki,Turku,Varsinais-Suomi,Asmandian aluetta ja Mynämäen Asemanseudun luo...,NaN,...,ajoittainen,NaN,NaN,503,023,02,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
2,375,MULTILINESTRING ((242236.65234453185 6712443.4...,I love Halinen -kulttuuriulkoilureitti (reitti 1),I love Halinen cultural excercise route (route 1),Jag älskar Hallis -kulturmotionsled (rutt 1),Turku,Turun seutu,Varsinais-Suomi,Reitti kulkee Halisten lähiön ja perinnemaisem...,Vid rutten kan du se både traditionell lantbru...,...,ei tietoa ylläpitoluokasta,NaN,NaN,853,23,02,False,Kävelyreitti/ulkoilureitti,4403,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
3,302296,"MULTILINESTRING ((224251.5 6734075.5, 224251.5...","Raukkaankosken rantapolku, kanoottien maihinno...",NaN,NaN,Mynämäki,Turku,Varsinais-Suomi,"Raukkaankosken rantapolku, pato, kanoottien ma...",NaN,...,ajoittainen,NaN,NaN,503,023,02,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
4,34,MULTILINESTRING ((264074.3350391999 6667861.29...,Vestlaxin luontopolku,Vestlax nature trail,Vestlax naturstig,Kemiönsaari,Turunmaa,Varsinais-Suomi,"Noin 3,8 kilometrin luontopolku",NaN,...,ei tietoa ylläpitoluokasta,NaN,NaN,322,21,02,False,Luontopolku,4404,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,422,MULTILINESTRING ((250170.94820000045 6742368.7...,Kuhankuono - Töykkälä reitti,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ei tietoa ylläpitoluokasta,NaN,NaN,NaN,NaN,NaN,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
233,466,"MULTILINESTRING ((235466.6974 6702696.6107, 23...",Käyränmäen kuntorata,NaN,NaN,Turku,Turun seutu,Varsinais-Suomi,NaN,NaN,...,ei tietoa ylläpitoluokasta,NaN,NaN,853,23,02,False,Kuntorata,4401,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
234,302223,"MULTILINESTRING ((212660.85000610352 6761273, ...",Vesitorninmäen voimaportaat,NaN,NaN,Laitila,Vakka-Suomi,Varsinais-Suomi,40 metriä pitkät kuntoportaat. Ei talvikunnoss...,NaN,...,ajoittainen,+358401234567,Hyvä,400,024,02,False,Kuntorata,4401,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
235,364,MULTILINESTRING ((244159.91363630912 6690154.6...,Mustikkapolku,Blueberry path,Blåbärstigen,Parainen,Turunmaa,Varsinais-Suomi,"2,8 km pitkä polku. Merkitty sinisillä merkinn...","2,8 km rutt. Blå markeringar. Terrängen är ber...",...,ei tietoa ylläpitoluokasta,NaN,Reitti on hyvin merkitty maastoon sinisillä vi...,445,21,02,False,Luontopolku,4404,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."


In [14]:
tarkistusdf[tarkistusdf['lipas_type_code'].isna()]["name_fi"]

Series([], Name: name_fi, dtype: object)

In [8]:
lipas_id_map = pd.read_csv("../../Data/virma_routes_gid_lipas_mapping (1).csv")

In [9]:
lipas_id_map

,gid,lipas_id
0,302297,619553
1,302295,619554
2,375,619555
3,302296,619556
4,34,619557
...,...,...
228,365,619781
229,466,619782
230,302223,619783
231,364,619784


In [10]:
tarkistusdf[~tarkistusdf['gid'].isin(lipas_id_map['gid'])]

,gid,geom,name_fi,name_en,name_se,municipali,subregion,region,info_fi,info_se,...,upkeepclas,upkeepinfo,shapeestim,munici_nro,subreg_nro,region_nro,ski_route,lipas_type_name,lipas_type_code,properties
68,423,MULTILINESTRING EMPTY,Kuhankuonon retkeilyreitistö - Karpalopolku,Kuhankuono hiking trails - cranberry path (Kar...,Kuhankuono vandringsledar - Tranbärstig (Karpa...,Pöytyä,Loimaan seutu,Varsinais-Suomi,Kuhankuonon reitistön polut kulkevat Etelä-Suo...,Kuhankuono vandringrutter går genom exceptione...,...,ajoittainen,NaN,NaN,636,25,02,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
126,8,MULTILINESTRING ((247302.0608000001 6743132.34...,Pukkipalo reitti,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ei tietoa ylläpitoluokasta,NaN,NaN,NaN,NaN,NaN,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
228,305536,MULTILINESTRING ((277864.1232084985 6685896.31...,Kalasuntin polku,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
232,422,MULTILINESTRING ((250170.94820000045 6742368.7...,Kuhankuono - Töykkälä reitti,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ei tietoa ylläpitoluokasta,NaN,NaN,NaN,NaN,NaN,False,Retkeilyreitti,4405,"{""accessibile?"": ""esteellinen"", ""toilet?"": fal..."
